In [ ]:
import pandas as pd


AR_CSV  = 'arkansas.csv'
CA_CSV  = 'california.csv'
OUT_CSV = 'stats_table2.csv'


ARKANSAS_LABELS = {
    1:  'Corn',
    2:  'Cotton',
    3:  'Rice',
    5:  'Soybeans',
    99: 'Others',
}
CALIFORNIA_LABELS = {
    3:   'Rice',
    36:  'Alfalfa',
    69:  'Grapes',
    75:  'Almonds',
    204: 'Pistachios',
    99:  'Others',
}


SAMPLES_PER_CLASS = 300
TRAIN_PER_CLASS   = 240

def compute_splits(total):
    reserved = min(total, SAMPLES_PER_CLASS)
    train    = min(reserved, TRAIN_PER_CLASS)
    val      = reserved - train
    testing  = total - reserved
    return train, val, testing

def build_state_rows(df, label_map, state_name):
    rows = []
    df = df.copy()
    df['cropland'] = df['cropland'].apply(lambda c: c if c in label_map else 99)
    counts = df['cropland'].value_counts()
    ordered_codes = [c for c in label_map if c != 99] + [99]
    for code in ordered_codes:
        name  = label_map[code]
        total = int(counts.get(code, 0))
        if total == 0:
            continue
        train, val, test = compute_splits(total)
        rows.append({
            'Study area':        state_name if len(rows) == 0 else '',
            'Class Name':        name,
            'Number of Samples': total,
            'Training':          train,
            'Validation':        val,
            'Testing':           test,
        })
    
    rows.append({
        'Study area':        '',
        'Class Name':        'All',
        'Number of Samples': sum(r['Number of Samples'] for r in rows),
        'Training':          sum(r['Training']          for r in rows),
        'Validation':        sum(r['Validation']        for r in rows),
        'Testing':           sum(r['Testing']           for r in rows),
    })
    return rows


all_rows = []

try:
    df_ar = pd.read_csv(AR_CSV)
    print(f'Loaded Arkansas  : {len(df_ar):,} rows')
    all_rows += build_state_rows(df_ar, ARKANSAS_LABELS, 'Arkansas')
except FileNotFoundError:
    print(f'Warning: {AR_CSV} not found - skipping Arkansas.')

try:
    df_ca = pd.read_csv(CA_CSV)
    print(f'Loaded California: {len(df_ca):,} rows')
    all_rows += build_state_rows(df_ca, CALIFORNIA_LABELS, 'California')
except FileNotFoundError:
    print(f'Warning: {CA_CSV} not found - skipping California.')

if not all_rows:
    print('No data loaded. Check your CSV paths.')
else:
    out_df = pd.DataFrame(all_rows, columns=[
        'Study area', 'Class Name', 'Number of Samples',
        'Training', 'Validation', 'Testing'
    ])
    out_df.to_csv(OUT_CSV, index=False)
    print(f'Saved -> {OUT_CSV}')
    display(out_df)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker


ar_df = pd.read_csv('arkansas.csv')
ca_df = pd.read_csv('california.csv')


AR_CROPS = {1: 'Corn', 2: 'Cotton', 3: 'Rice', 5: 'Soybeans', 99: 'Others'}
CA_CROPS = {3: 'Rice', 36: 'Alfalfa', 69: 'Grapes', 75: 'Almonds', 204: 'Pistachios', 99: 'Others'}



N_COMPOSITES = 36
DOY = np.arange(1, 366, 365 / N_COMPOSITES).astype(int)   

def compute_mean_ndvi(df, crop_map):
    
    nir_cols = [f'{t}_B8'  for t in range(N_COMPOSITES)]
    red_cols = [f'{t}_B4'  for t in range(N_COMPOSITES)]

    nir = df[nir_cols].values.astype(float)
    red = df[red_cols].values.astype(float)

    nir[nir == 0] = np.nan
    red[red == 0] = np.nan

    ndvi = (nir - red) / (nir + red + 1e-9)

    result = {}
    for code, name in crop_map.items():
        mask = df['cropland'] == code
        if mask.sum() == 0:
            continue
        result[name] = np.nanmean(ndvi[mask], axis=0)
    return result

ar_ndvi = compute_mean_ndvi(ar_df, AR_CROPS)
ca_ndvi = compute_mean_ndvi(ca_df, CA_CROPS)




MARKERS = ['o', 's', '^', 'D', 'v', 'P']
COLORS  = ['#2ca02c', '#d62728', '#ff7f0e', '#1f77b4', '#9467bd', '#8c564b']

fig, axes = plt.subplots(2, 1, figsize=(9, 8), sharey=False)

for ax, ndvi_dict, title in zip(
        axes,
        [ar_ndvi, ca_ndvi],
        ['(a) Arkansas', '(b) California']):

    for i, (crop, vals) in enumerate(ndvi_dict.items()):
        
        valid = ~np.isnan(vals)
        ax.plot(DOY[valid], vals[valid],
                marker=MARKERS[i % len(MARKERS)],
                color=COLORS[i % len(COLORS)],
                linewidth=1.5,
                markersize=5,
                label=crop)

    ax.set_title(title, fontsize=12, loc='left', pad=6)
    ax.set_xlabel('Day of Year', fontsize=10)
    ax.set_ylabel('Mean NDVI Value', fontsize=10)
    ax.set_xlim(0, 370)
    ax.set_ylim(0, 1)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(50))
    ax.grid(False)
    ax.legend(loc='upper right', fontsize=8, ncol=2, frameon=True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout(pad=2.5)
plt.savefig('ndvi_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()